In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable, Literal

import numpy as np
import pandas as pd
import shap
from dotenv import load_dotenv
from entsoe.entsoe import EntsoePandasClient
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

load_dotenv()

NORWAY_ZONES: tuple[str, ...] = ("NO_1", "NO_2", "NO_3", "NO_4", "NO_5")

CROSSBORDER_PAIRS: tuple[tuple[str, str], ...] = (
    ("NO_1", "SE_3"),
    ("NO_2", "SE_4"),
    ("NO_2", "DK_1"),
    ("NO_4", "SE_2"),
    ("NO_5", "GB"),
    ("NO_2", "NL"),
)

RENEWABLE_KEYWORDS: tuple[str, ...] = ("Wind", "Solar", "Hydro", "Biomass", "Geothermal")


@dataclass(frozen=True)
class Config:
    """Runtime configuration for data retrieval, feature engineering, and evaluation."""

    api_key: str
    output_dir: Path = Path("./entsoe_no_experiment")
    entsoe_tz: str = "Europe/Brussels"
    local_tz: str = "Europe/Oslo"
    months_back: int = 3
    frequency: str = "H"
    test_days: int = 14
    spike_quantile: float = 0.95
    random_state: int = 42
    min_train_rows: int = 24 * 60
    min_test_rows: int = 24 * 7
    shap_background_size: int = 2000


class EntsoeDataLoader:
    """Download ENTSO-E data for Norwegian zones and relevant cross-border flows."""

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.client = EntsoePandasClient(api_key=cfg.api_key)
        self.start_utc, self.end_utc = self._time_window()

    def _time_window(self) -> tuple[pd.Timestamp, pd.Timestamp]:
        end_utc = pd.Timestamp(datetime.now(), tz=self.cfg.entsoe_tz).floor(self.cfg.frequency)
        start_utc = (end_utc - pd.DateOffset(months=self.cfg.months_back)).floor(self.cfg.frequency)
        return start_utc, end_utc

    @staticmethod
    def _as_series(obj: object, series_name: str) -> pd.Series:
        if isinstance(obj, pd.DataFrame):
            if obj.shape[1] != 1:
                raise ValueError(f"Expected a single column for '{series_name}', got {list(obj.columns)}")
            series = obj.iloc[:, 0].copy()
            series.name = series_name
            return series
        if isinstance(obj, pd.Series):
            series = obj.copy()
            series.name = series_name
            return series
        raise TypeError(f"Expected pandas Series or 1-col DataFrame for '{series_name}', got {type(obj)}")

    @staticmethod
    def _as_frame(obj: object, prefix: str, suffix: str) -> pd.DataFrame:
        if isinstance(obj, pd.Series):
            frame = obj.to_frame()
        elif isinstance(obj, pd.DataFrame):
            frame = obj.copy()
        else:
            raise TypeError(f"Expected pandas Series or DataFrame for '{prefix}', got {type(obj)}")

        frame.columns = [f"{prefix}{c}{suffix}" for c in frame.columns]
        return frame

    def fetch_zone_data(self, zone: str) -> pd.DataFrame:
        pieces: list[pd.DataFrame] = []

        prices = self._as_series(
            self.client.query_day_ahead_prices(zone, start=self.start_utc, end=self.end_utc),
            "price_eur_mwh",
        )
        pieces.append(prices.to_frame())

        try:
            load_actual = self._as_series(
                self.client.query_load(zone, start=self.start_utc, end=self.end_utc),
                "load_mw",
            )
            pieces.append(load_actual.to_frame())
        except Exception:
            pass

        try:
            load_forecast = self._as_series(
                self.client.query_load_forecast(zone, start=self.start_utc, end=self.end_utc),
                "load_forecast_mw",
            )
            pieces.append(load_forecast.to_frame())
        except Exception:
            pass

        try:
            generation_by_type = self.client.query_generation(zone, start=self.start_utc, end=self.end_utc)
            generation_frame = self._as_frame(generation_by_type, prefix="gen_", suffix="_mw")
            pieces.append(generation_frame)
        except Exception:
            pass

        try:
            wind_solar_forecast = self.client.query_wind_and_solar_forecast(
                zone, start=self.start_utc, end=self.end_utc
            )
            wind_solar_frame = self._as_frame(wind_solar_forecast, prefix="ws_forecast_", suffix="_mw")
            pieces.append(wind_solar_frame)
        except Exception:
            pass

        merged = pd.concat(pieces, axis=1).sort_index()
        return merged

    def fetch_crossborder_flows(self, pairs: Iterable[tuple[str, str]]) -> pd.DataFrame:
        flow_series: list[pd.Series] = []
        for from_area, to_area in pairs:
            column_name = f"flow_{from_area}_to_{to_area}_mw"
            try:
                s = self._as_series(
                    self.client.query_crossborder_flows(from_area, to_area, start=self.start_utc, end=self.end_utc),
                    column_name,
                )
                flow_series.append(s)
            except Exception:
                continue

        if not flow_series:
            return pd.DataFrame()

        return pd.concat(flow_series, axis=1).sort_index()


class FeatureBuilder:
    """Transform raw zone-level data into leakage-safe, ML-ready features."""

    def __init__(
        self,
        local_tz: str = "Europe/Oslo",
        frequency: str = "H",
        renewable_keywords: tuple[str, ...] = RENEWABLE_KEYWORDS,
        price_lags: tuple[int, ...] = (1, 2, 3, 24, 48, 168),
        price_roll_windows: tuple[int, ...] = (3, 6, 12, 24, 72, 168),
        load_lags: tuple[int, ...] = (1, 3, 24, 168),
        load_roll_windows: tuple[int, ...] = (3, 6, 12, 24, 168),
        flow_lags: tuple[int, ...] = (1, 24, 168),
    ) -> None:
        self.local_tz = local_tz
        self.frequency = frequency
        self.renewable_keywords = renewable_keywords
        self.price_lags = price_lags
        self.price_roll_windows = price_roll_windows
        self.load_lags = load_lags
        self.load_roll_windows = load_roll_windows
        self.flow_lags = flow_lags

    def build(self, raw: pd.DataFrame) -> pd.DataFrame:
        if raw.empty:
            return pd.DataFrame()

        df = raw.copy()
        df = self._ensure_hourly_index(df)

        if "price_eur_mwh" not in df.columns:
            return pd.DataFrame()

        self._add_calendar_features(df)
        self._add_price_features(df)
        self._add_load_features(df)
        self._add_generation_features(df)
        self._add_wind_solar_forecast_features(df)
        self._add_flow_features(df)

        df = df.dropna(subset=["price_eur_mwh"]).copy()
        return df

    def _ensure_hourly_index(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.sort_index()
        if not isinstance(df.index, pd.DatetimeIndex) or df.index.tz is None:
            raise ValueError("Input index must be timezone-aware (ENTSO-E returns tz-aware timestamps).")

        df.index = df.index.tz_convert(self.local_tz)
        full_index = pd.date_range(df.index.min(), df.index.max(), freq=self.frequency, tz=self.local_tz)
        return df.reindex(full_index)

    @staticmethod
    def _add_calendar_features(df: pd.DataFrame) -> None:
        idx = pd.DatetimeIndex(df.index)
        df["hour"] = idx.hour
        df["day_of_week"] = idx.dayofweek
        df["month"] = idx.month
        df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    def _add_price_features(self, df: pd.DataFrame) -> None:
        price = df["price_eur_mwh"]
        for lag in self.price_lags:
            df[f"price_eur_mwh_lag_{lag}"] = price.shift(lag)

        shifted = price.shift(1)
        for window in self.price_roll_windows:
            df[f"price_eur_mwh_roll_mean_{window}"] = shifted.rolling(window, min_periods=window).mean()
            df[f"price_eur_mwh_roll_std_{window}"] = shifted.rolling(window, min_periods=window).std()

    def _add_load_features(self, df: pd.DataFrame) -> None:
        if "load_mw" in df.columns:
            load_actual = df["load_mw"]
            for lag in self.load_lags:
                df[f"load_mw_lag_{lag}"] = load_actual.shift(lag)

            shifted = load_actual.shift(1)
            for window in self.load_roll_windows:
                df[f"load_mw_roll_mean_{window}"] = shifted.rolling(window, min_periods=window).mean()
                df[f"load_mw_roll_std_{window}"] = shifted.rolling(window, min_periods=window).std()

        if "load_forecast_mw" in df.columns:
            df["load_forecast_mw_lag_1"] = df["load_forecast_mw"].shift(1)

    def _add_generation_features(self, df: pd.DataFrame) -> None:
        generation_columns = [c for c in df.columns if c.startswith("gen_") and c.endswith("_mw")]
        if not generation_columns:
            return

        df["gen_total_mw"] = df[generation_columns].sum(axis=1)

        renewable_columns = [
            c for c in generation_columns if any(k.lower() in c.lower() for k in self.renewable_keywords)
        ]
        if renewable_columns:
            df["gen_renewable_mw"] = df[renewable_columns].sum(axis=1)
            df["gen_renewable_share"] = df["gen_renewable_mw"] / df["gen_total_mw"].replace(0, np.nan)

        for col in ("gen_total_mw", "gen_renewable_mw", "gen_renewable_share"):
            if col in df.columns:
                df[f"{col}_lag_1"] = df[col].shift(1)

    @staticmethod
    def _add_wind_solar_forecast_features(df: pd.DataFrame) -> None:
        forecast_columns = [c for c in df.columns if c.startswith("ws_forecast_") and c.endswith("_mw")]
        if not forecast_columns:
            return

        df["ws_forecast_total_mw"] = df[forecast_columns].sum(axis=1)
        df["ws_forecast_total_mw_lag_1"] = df["ws_forecast_total_mw"].shift(1)

    def _add_flow_features(self, df: pd.DataFrame) -> None:
        flow_columns = [c for c in df.columns if c.startswith("flow_") and c.endswith("_mw")]
        if not flow_columns:
            return

        df["flow_total_mw"] = df[flow_columns].sum(axis=1)

        for lag in self.flow_lags:
            df[f"flow_total_mw_lag_{lag}"] = df["flow_total_mw"].shift(lag)

        for col in flow_columns:
            df[f"{col}_lag_1"] = df[col].shift(1)


class FeaturePreprocessor:
    """Training-consistent preprocessing (ffill + median imputation + inf handling + numeric-only)."""

    def __init__(self) -> None:
        self.numeric_feature_names: list[str] = []
        self.median_values: pd.Series | None = None

    def fit(self, X: pd.DataFrame) -> FeaturePreprocessor:
        X_clean = X.copy().ffill()
        X_clean = X_clean.replace([np.inf, -np.inf], np.nan)

        numeric_columns = [c for c in X_clean.columns if pd.api.types.is_numeric_dtype(X_clean[c])]
        X_numeric = X_clean[numeric_columns]

        medians = X_numeric.median(numeric_only=True)
        self.numeric_feature_names = list(X_numeric.columns)
        self.median_values = medians
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.median_values is None:
            raise ValueError("Preprocessor is not fitted.")

        X_clean = X.copy().ffill()
        X_clean = X_clean.replace([np.inf, -np.inf], np.nan)

        X_numeric = X_clean.reindex(columns=self.numeric_feature_names)
        X_filled = X_numeric.fillna(self.median_values)
        return X_filled

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


def time_split(df: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    if df.empty:
        return df, df
    cutoff = df.index.max() - pd.Timedelta(days=cfg.test_days)
    train = df[df.index <= cutoff].copy()
    test = df[df.index > cutoff].copy()
    return train, test


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {"rmse": rmse, "mae": mae, "r2": r2}


def spike_threshold_from_train(train_target: pd.Series, quantile: float) -> float:
    if train_target.empty:
        raise ValueError("Train target is empty; cannot compute spike threshold.")
    return float(train_target.quantile(quantile))


def add_spike_labels(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    out = df.copy()
    out["spike_threshold"] = threshold
    out["is_spike"] = (out["price_eur_mwh"] >= threshold).astype(int)
    return out


def select_feature_sets(feature_df: pd.DataFrame, target_column: str) -> dict[str, list[str]]:
    all_columns = [c for c in feature_df.columns if c != target_column]
    autoregressive_features = [
        c
        for c in all_columns
        if c.startswith("price_eur_mwh_") and ("lag_" in c or "roll_" in c)
    ]
    excluded = {"is_spike", "spike_threshold"}
    exogenous_features = [c for c in all_columns if c not in excluded and c not in autoregressive_features]
    full_features = sorted(set(autoregressive_features + exogenous_features))
    return {
        "autoregressive": autoregressive_features,
        "exogenous": exogenous_features,
        "full": full_features,
    }


def train_mean_models(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_names: list[str],
    cfg: Config,
    target_column: str = "price_eur_mwh",
) -> tuple[pd.DataFrame, dict[str, dict[str, float]], dict[str, object]]:
    if train_df.empty or test_df.empty:
        raise ValueError("Train/test split produced empty dataframes.")
    if not feature_names:
        raise ValueError("No features selected.")

    X_train_raw = train_df[feature_names].copy()
    y_train = train_df[target_column].copy()
    X_test_raw = test_df[feature_names].copy()
    y_test = test_df[target_column].copy()

    preprocessor = FeaturePreprocessor()
    X_train = preprocessor.fit_transform(X_train_raw)
    X_test = preprocessor.transform(X_test_raw)

    lgbm = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=0.0,
        random_state=cfg.random_state,
    )
    lgbm.fit(X_train, y_train)

    xgb = XGBRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=cfg.random_state,
        n_jobs=-1,
    )
    xgb.fit(X_train, y_train)

    pred_lgbm = lgbm.predict(X_test)
    pred_xgb = xgb.predict(X_test)

    metrics = {
        "lgbm": regression_metrics(y_test.to_numpy(), pred_lgbm),
        "xgb": regression_metrics(y_test.to_numpy(), pred_xgb),
    }

    predictions = pd.DataFrame(
        {"y_true": y_test, "pred_lgbm": pred_lgbm, "pred_xgb": pred_xgb},
        index=test_df.index,
    )

    artifacts: dict[str, object] = {
        "feature_names": list(X_train.columns),
        "preprocessor": preprocessor,
        "lgbm": lgbm,
        "xgb": xgb,
    }
    return predictions, metrics, artifacts


def train_quantile_models(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_names: list[str],
    cfg: Config,
    quantiles: tuple[float, ...] = (0.1, 0.5, 0.9),
    target_column: str = "price_eur_mwh",
) -> pd.DataFrame:
    X_train_raw = train_df[feature_names].copy()
    y_train = train_df[target_column].copy()
    X_test_raw = test_df[feature_names].copy()
    y_test = test_df[target_column].copy()

    preprocessor = FeaturePreprocessor()
    X_train = preprocessor.fit_transform(X_train_raw)
    X_test = preprocessor.transform(X_test_raw)

    out = pd.DataFrame(index=test_df.index)
    for q in quantiles:
        model = LGBMRegressor(
            objective="quantile",
            alpha=q,
            n_estimators=2500,
            learning_rate=0.02,
            num_leaves=64,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=cfg.random_state,
        )
        model.fit(X_train, y_train)
        out[f"q{int(q * 100):02d}"] = model.predict(X_test)

    out["y_true"] = y_test.to_numpy()
    return out


def shap_global_importance(
    model: object,
    X: pd.DataFrame,
    output_path: Path,
    background_size: int,
    max_display: int = 25,
) -> pd.DataFrame:
    if X.empty or X.shape[1] == 0:
        raise ValueError("SHAP input matrix is empty.")

    background = X
    if len(X) > background_size:
        background = X.sample(background_size, random_state=42)

    explainer = shap.TreeExplainer(
        model,
        data=background,
        feature_perturbation="interventional",
        model_output="raw",
    )
    shap_values = explainer(X, check_additivity=False)

    values = np.asarray(shap_values.values)
    if values.ndim == 3:
        values = values[:, :, 0]

    mean_abs = np.abs(values).mean(axis=0)
    importance = (
        pd.DataFrame({"feature": X.columns, "mean_abs_shap": mean_abs})
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )

    shap.plots.bar(shap_values, max_display=max_display, show=False)
    import matplotlib.pyplot as plt

    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

    return importance


def filter_flow_pairs_for_zone(zone: str, all_pairs: Iterable[tuple[str, str]]) -> list[tuple[str, str]]:
    return [(a, b) for (a, b) in all_pairs if a == zone or b == zone]


def run_for_feature_set(
    zone: str,
    feature_df: pd.DataFrame,
    cfg: Config,
    feature_set: Literal["autoregressive", "exogenous", "full"],
) -> tuple[pd.DataFrame, list[dict[str, object]], pd.DataFrame]:
    train_df, test_df = time_split(feature_df, cfg)

    if len(train_df) < cfg.min_train_rows or len(test_df) < cfg.min_test_rows:
        raise ValueError(f"Not enough data. train={len(train_df)}, test={len(test_df)}")

    threshold = spike_threshold_from_train(train_df["price_eur_mwh"], cfg.spike_quantile)
    train_df = add_spike_labels(train_df, threshold)
    test_df = add_spike_labels(test_df, threshold)

    feature_sets = select_feature_sets(feature_df, target_column="price_eur_mwh")
    selected_features = feature_sets[feature_set]
    if not selected_features:
        raise ValueError(f"Feature set '{feature_set}' is empty.")

    predictions, metrics, artifacts = train_mean_models(
        train_df=train_df,
        test_df=test_df,
        feature_names=selected_features,
        cfg=cfg,
    )
    quantile_predictions = train_quantile_models(
        train_df=train_df,
        test_df=test_df,
        feature_names=selected_features,
        cfg=cfg,
    )

    combined = predictions.join(quantile_predictions.drop(columns=["y_true"]), how="left")
    combined["zone"] = zone
    combined["feature_set"] = feature_set
    combined["spike_threshold"] = threshold
    combined["is_spike"] = test_df["is_spike"].astype(int)

    metric_rows: list[dict[str, object]] = []
    for model_name, m in metrics.items():
        metric_rows.append({"zone": zone, "feature_set": feature_set, "model": model_name, **m})

    preprocessor: FeaturePreprocessor = artifacts["preprocessor"]
    X_train_for_shap = preprocessor.transform(train_df[artifacts["feature_names"]])
    shap_plot_path = cfg.output_dir / f"{zone}__{feature_set}__shap_bar.png"
    shap_importance = shap_global_importance(
        artifacts["lgbm"],
        X_train_for_shap,
        output_path=shap_plot_path,
        background_size=cfg.shap_background_size,
    )
    shap_importance["zone"] = zone
    shap_importance["feature_set"] = feature_set

    return combined, metric_rows, shap_importance

/tmp/ipykernel_15133/2434568039.py:61: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  end_utc = pd.Timestamp(datetime.now(), tz=self.cfg.entsoe_tz).floor(self.cfg.frequency)
/tmp/ipykernel_15133/2434568039.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  start_utc = (end_utc - pd.DateOffset(months=self.cfg.months_back)).floor(self.cfg.frequency)
/tmp/ipykernel_15133/2434568039.py:205: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(df.index.min(), df.index.max(), freq=self.frequency, tz=self.local_tz)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000721 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 18
[LightGBM] [Info] Start training from score 60.161714
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000836 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 18
[LightGBM] [Info] Start training from score 19.690001
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000505 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 18
[LightGBM] [Info] Start trai

100%|===================| 8421/8425 [15:17<00:00]        

[OK] NO_1/autoregressive: rows=336 shap_features=18
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002017 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9515
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 43
[LightGBM] [Info] Start training from score 60.161714
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001045 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9515
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 43
[LightGBM] [Info] Start training from score 19.690001
[LightGBM] [Warning] Found whitespace in

100%|===================| 8423/8425 [16:46<00:00]        

...

100%|===================| 8417/8425 [17:39<00:01]        

[OK] NO_1/full: rows=336 shap_features=62
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000409 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 18
[LightGBM] [Info] Start training from score 67.070871


/tmp/ipykernel_15133/2434568039.py:205: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(df.index.min(), df.index.max(), freq=self.frequency, tz=self.local_tz)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000638 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 18
[LightGBM] [Info] Start training from score 33.683998
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000934 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 18
[LightGBM] [Info] Start training from score 67.190002
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001010 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 8425, number of used features: 18
[LightGBM] [Info] Start trai

 17%|===                 | 1443/8425 [02:45<13:18]       

In [ ]:
cfg = Config(api_key=os.getenv("ENTSOE_API_KEY", ""))
if not cfg.api_key:
    raise ValueError("Missing ENTSOE_API_KEY in environment (e.g., .env).")

cfg.output_dir.mkdir(parents=True, exist_ok=True)

data_loader = EntsoeDataLoader(cfg)
feature_builder = FeatureBuilder(local_tz=cfg.local_tz, frequency=cfg.frequency)

all_zone_raw_frames: list[pd.DataFrame] = []

full_flows = data_loader.fetch_crossborder_flows(CROSSBORDER_PAIRS)

for zone in NORWAY_ZONES:
    zone_raw = data_loader.fetch_zone_data(zone)

    zone_pairs = filter_flow_pairs_for_zone(zone, CROSSBORDER_PAIRS)
    zone_flow_columns = [f"flow_{a}_to_{b}_mw" for (a, b) in zone_pairs]
    zone_flows = full_flows.reindex(columns=[c for c in zone_flow_columns if c in full_flows.columns])

    zone_merged = zone_raw.join(zone_flows, how="left")
    zone_prefixed = zone_merged.copy()
    zone_prefixed.columns = [f"{zone}__{c}" for c in zone_prefixed.columns]
    all_zone_raw_frames.append(zone_prefixed)

raw_all = pd.concat(all_zone_raw_frames, axis=1).sort_index()
if not raw_all.empty and isinstance(raw_all.index, pd.DatetimeIndex):
    raw_all.index = raw_all.index.tz_convert(cfg.local_tz)

raw_path = cfg.output_dir / f"raw_{cfg.months_back}m.parquet"
raw_all.to_parquet(raw_path)

all_predictions: list[pd.DataFrame] = []
all_metric_rows: list[dict[str, object]] = []
all_shap_frames: list[pd.DataFrame] = []

for zone in NORWAY_ZONES:
    zone_columns = [c for c in raw_all.columns if c.startswith(f"{zone}__")]
    zone_raw = raw_all[zone_columns].copy()
    zone_raw.columns = [c.split("__", 1)[1] for c in zone_raw.columns]

    feature_df = feature_builder.build(zone_raw)
    if feature_df.empty:
        print(f"[SKIP] {zone}: no usable feature table.")
        continue

    feature_df.to_parquet(cfg.output_dir / f"{zone}__features.parquet")

    for feature_set in ("autoregressive", "exogenous", "full"):
        try:
            predictions, metric_rows, shap_frame = run_for_feature_set(
                zone=zone,
                feature_df=feature_df,
                cfg=cfg,
                feature_set=feature_set,
            )
            all_predictions.append(predictions)
            all_metric_rows.extend(metric_rows)
            all_shap_frames.append(shap_frame)

            predictions.to_parquet(cfg.output_dir / f"{zone}__{feature_set}__predictions.parquet")
            print(f"[OK] {zone}/{feature_set}: rows={len(predictions)} shap_features={len(shap_frame)}")
        except Exception as e:
            print(f"[SKIP] {zone}/{feature_set}: {type(e).__name__}: {e}")

if all_predictions:
    pd.concat(all_predictions).sort_index().to_parquet(cfg.output_dir / "all_zones__predictions.parquet")

if all_metric_rows:
    pd.DataFrame(all_metric_rows).sort_values(["zone", "feature_set", "model"]).to_csv(
        cfg.output_dir / "metrics.csv",
        index=False,
    )

if all_shap_frames:
    (
        pd.concat(all_shap_frames)
        .sort_values(["zone", "feature_set", "mean_abs_shap"], ascending=[True, True, False])
        .to_csv(cfg.output_dir / "shap_importance.csv", index=False)
    )

print(f"Saved raw: {raw_path}")
if all_metric_rows:
    print(f"Saved metrics: {cfg.output_dir / 'metrics.csv'}")
if all_predictions:
    print(f"Saved predictions: {cfg.output_dir / 'all_zones__predictions.parquet'}")
if all_shap_frames:
    print("Saved per-zone SHAP bar plots (PNG) for each feature set.")